# Journey 4 — Prepare SWAN grids and input data

**Learning goals:** Connect a SWAN grid to bathymetry, wind, and wave-boundary data, and understand where data extraction occurs in the workflow.

**Prerequisites:** [Journey 3](../journey_03_swan_declarative/), xarray basics, and the SWAN plugin.


## Shared data concept

This lesson is the concrete SWAN example of the shared [Rompy data concepts](../../data-concepts/). `SwanDataGrid` and `Boundnest1` describe source plugins, variables, model domain, and time window together. Rompy selects the required slice and writes SWAN-native files; the upstream dataset does not need to travel with the run description.

## Why this matters: model-ready data preparation

**Without Rompy:** each SWAN source can require separate extraction, interpolation, and model-format conversion steps.

**With Rompy:** `SwanDataGrid` and boundary components connect source variables to the SWAN grid and period, then prepare model-native inputs. The modeller still chooses the dataset, variables, coordinates, interpolation, and quality checks; Rompy keeps the repetitive transformation consistent.


## 1. Define the model grid

The grid is the spatial contract shared by model configuration and data preparation.


In [ ]:
from rompy_swan.grid import SwanGrid

grid = SwanGrid(
    x0=115.0, y0=-32.0, rot=0.0,
    dx=0.25, dy=0.25, nx=9, ny=7,
)
print(grid.bbox())


## 2. Prepare bathymetry

A `SwanDataGrid` describes how a source variable becomes SWAN bottom data. In production, the source may be a local NetCDF file, a catalog, or another registered source. For the render-only path, the important concept is the transformation contract.


In [ ]:
from rompy_swan.data import SwanDataGrid
from rompy.core.source import SourceFile

bottom = SwanDataGrid(
    var="bottom",
    source=SourceFile(uri="path/to/local/bathymetry.nc"),
    z1="elevation",
    fac=-1,
    coords={"x": "lon", "y": "lat"},
)
print(bottom.var, bottom.z1)


## 3. Add wind and wave-boundary data

Wind is represented as a gridded input, while offshore wave information is commonly represented as boundary spectra. Both are connected to the SWAN configuration through interfaces.


In [ ]:
from rompy_swan.data import SwanDataGrid
from rompy_swan.boundary import Boundnest1

wind = SwanDataGrid(
    var="wind",
    source=SourceFile(uri="path/to/local/wind.nc"),
    z1="u10", z2="v10",
    coords={"x": "longitude", "y": "latitude"},
)
boundary = Boundnest1(
    id="offshore",
    source=SourceFile(uri="path/to/local/wave_boundary.nc"),
    sel_method="idw",
    sel_method_kwargs={"tolerance": 4},
)
print(wind.var, boundary.id)


## 4. Connect data through interfaces

The data objects are not standalone output files. They become model inputs when passed to SWAN’s data and boundary interfaces. Data filtering, coordinate mapping, cropping, and interpolation should be inspected before generating the workspace.


In [ ]:
from rompy_swan.interface import DataInterface, BoundaryInterface

inpgrid = DataInterface(bottom=bottom, input=[wind])
boundary_interface = BoundaryInterface(kind=boundary)
print(inpgrid)
print(boundary_interface)


## Checkpoint

The model domain and its three main input families are now explicit. Replace the example URIs with compatible local data when executing this lesson. The documentation build does not fetch or process these files.

**Next:** [Configure SWAN components and outputs](../journey_05_swan_components/).

**Deep dives:** [Boundary nested example](boundary/boundnest1/) and the [general data-source guide](https://rom-py.github.io/rompy/reference/source/).


## Visual verification: grid and forcing

This small teaching field makes the relationship between the SWAN grid, bathymetry, and wind forcing concrete. In a real run the same objects can read NetCDF or catalog sources.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(115, 117, 9)
y = np.linspace(-32, -30.5, 7)
xx, yy = np.meshgrid(x, y)
bathymetry = 8 + 0.8 * (xx - x.min()) + 1.2 * (yy - y.min())
wind_u = np.full_like(xx, 7.0)
wind_v = np.full_like(yy, 2.0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].pcolormesh(xx, yy, bathymetry, shading="auto", cmap="Blues")
axes[0].set_title("Bathymetry on the SWAN grid")
axes[0].set_xlabel("longitude"); axes[0].set_ylabel("latitude")
axes[1].pcolormesh(xx, yy, np.hypot(wind_u, wind_v), shading="auto", cmap="viridis")
axes[1].quiver(xx, yy, wind_u, wind_v, color="white")
axes[1].set_title("Wind source mapped to the domain")
axes[1].set_xlabel("longitude")
plt.show()


The important check is not the appearance of the plot: it is that the fields cover the intended domain, use the expected units and coordinates, and have the time coverage required by the `TimeRange`. Rompy can perform configured selection and conversion, but it cannot decide whether a wind field is scientifically appropriate.


## Real fixture comparison

The repository’s shared fixtures make the source fields concrete without requiring a network download. GEBCO provides elevation; ERA5 provides wind components. The source coordinates and units remain scientific assumptions that must be checked before production use.


In [ ]:
from pathlib import Path
import xarray as xr

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "tests" / "data").is_dir())
gebco = xr.open_dataset(root / "tests/data/gebco-1deg.nc")
era5 = xr.open_dataset(root / "tests/data/era5-20230101.nc")
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
gebco.elevation.sel(lon=slice(110, 120), lat=slice(-40, -20)).plot(ax=axes[0], cmap="terrain")
axes[0].set_title("GEBCO source elevation")
era5.u10.isel(time=0).plot(ax=axes[1], cmap="coolwarm")
axes[1].quiver(era5.longitude, era5.latitude, era5.u10.isel(time=0), era5.v10.isel(time=0), color="black", scale=250)
axes[1].set_title("ERA5 source wind")
np.hypot(era5.u10, era5.v10).mean(dim=("latitude", "longitude")).plot(ax=axes[2])
axes[2].set_title("ERA5 time coverage")
plt.show()
print("GEBCO:", dict(gebco.sizes), "ERA5:", dict(era5.sizes))
gebco.close(); era5.close()


These source plots verify coverage, not SWAN model skill. Rompy’s `fac=-1` bathymetry conversion, latitude ordering, wind units, grid interpolation, and spectral boundary convention must be reviewed before operational use. The placeholder URI cells above remain the portable API pattern; generation requires compatible local wave-boundary data.

## Rompy processing: SWAN grid, wind, and spectral boundary files

The three SWAN data objects now perform their transformations. Rompy writes the SWAN bottom grid, wind grid, and nested spectral boundary file into a temporary workspace; the returned filenames are the hand-off to the SWAN `INPUT` commands.

In [ ]:
from tempfile import TemporaryDirectory
from rompy.core.filters import Filter
from rompy.core.source import SourceFile
from rompy.core.time import TimeRange
from rompy_swan.boundary import Boundnest1
from rompy_swan.data import SwanDataGrid

source_grid = SwanGrid(x0=115.0, y0=-32.0, rot=0.0, dx=0.25, dy=0.25, nx=9, ny=7)
source_time = TimeRange(start="2023-01-01", end="2023-01-01T06:00", interval="1h")
bottom_source = SwanDataGrid(var="bottom", source=SourceFile(uri=root / "tests/data/gebco-1deg.nc"), z1="elevation", fac=-1, coords={"x":"lon", "y":"lat"}, crop_data=False)
wind_source = SwanDataGrid(var="wind", source=SourceFile(uri=root / "tests/data/era5-20230101.nc"), z1="u10", z2="v10", coords={"x":"longitude", "y":"latitude"}, crop_data=False, filter=Filter(sort={"coords":["latitude"]}))
boundary_source = Boundnest1(id="westaus", source=SourceFile(uri=root / "tests/data/aus-20230101.nc"), sel_method="idw", sel_method_kwargs={"tolerance":4})
with TemporaryDirectory() as output:
    outputs = {"bottom": bottom_source.get(output, source_grid, source_time), "wind": wind_source.get(output, source_grid, source_time), "boundary": boundary_source.get(output, source_grid, source_time)}
    for name, value in outputs.items(): print(f"Rompy {name} output:", value)
    assert (Path(output) / "bottom.grd").is_file()
    assert (Path(output) / "wind.grd").is_file()
    assert Path(outputs["boundary"][0]).is_file()
    print("Generated SWAN data files: bottom.grd, wind.grd, westaus.bnd")
    print("The corresponding INPGRID/READINP commands are exposed by the data objects when assembled in SwanConfig.")
    from scripts.example_outputs import report
    print("Reproducibility report:", report(output, tier="configuration-only"))
